### ***Our Architecture***
> We'll build exactly this:

Raw Data
     │
     ▼
ColumnTransformer
     │
     ├──────────────┐
     │              │
     ▼              ▼
Numeric         Categorical
Pipeline         Pipeline
     │              │
     ▼              ▼
Imputer         Imputer
     │              │
     ▼              ▼
Scaler          OneHotEncoder
      \            /
       \          /
        ──────────
             │
             ▼
 Logistic Regression
             │
             ▼
 Prediction

### ***However, before writing a single line of code, let's understand one important concept***




>  Before Pipeline, We Need to Understand `ColumnTransformer`

-  Why?

    > Our dataset has two types of columns:

| Numerical  | Categorical |
| ---------- | ----------- |
| Age        | Gender      |
| Salary     | City        |
| Experience |             |

Now think...

Can we apply **OneHotEncoder** to `Age` or `Salary`?

❌ No.

Can we apply **StandardScaler** to `Gender` or `City`?

❌ No.

Different columns need different preprocessing.

Without Pipeline, we manually wrote:

```text
Numerical Columns
      ↓
SimpleImputer
      ↓
StandardScaler
```

and

```text
Categorical Columns
      ↓
SimpleImputer
      ↓
OneHotEncoder
```

Then we manually combined them.

That manual work is exactly what **ColumnTransformer** automates.

---

# Relationship Between ColumnTransformer and Pipeline

```text
                Pipeline
                   │
        ┌──────────┴──────────┐
        │                     │
ColumnTransformer         LogisticRegression
        │
   ┌────┴─────┐
   │          │
Numeric    Categorical
Pipeline     Pipeline
```

So:

* **ColumnTransformer** decides **which preprocessing goes to which columns**.
* **Pipeline** decides **the order of the complete ML workflow**.

---

# Our Architecture

We'll build exactly this:

```text
Raw Data
     │
     ▼
ColumnTransformer
     │
     ├──────────────┐
     │              │
     ▼              ▼
Numeric         Categorical
Pipeline         Pipeline
     │              │
     ▼              ▼
Imputer         Imputer
     │              │
     ▼              ▼
Scaler          OneHotEncoder
      \            /
       \          /
        ──────────
             │
             ▼
 Logistic Regression
             │
             ▼
 Prediction
```


# ***Step 1: Import Libraries and Import dataset***

In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression

In [31]:
df = pd.read_csv('Dataset.csv')

In [32]:
df.head()

,Age,Salary,Experience,Gender,City,Purchased
0,56.0,60794.0,33,Female,Hyderabad,1
1,46.0,99459.0,18,Female,Delhi,0
2,32.0,70192.0,20,Female,Bangalore,0
3,25.0,47703.0,33,Female,Mumbai,0
4,NaN,104193.0,16,Female,Mumbai,1


In [33]:
# Remove Duplicate Rows
df.duplicated().sum()

35

In [34]:
df = df.drop_duplicates()

# ***Step 2: Separate Features and Target***
> First tell the computer which columns are numerical and which are categorical.

In [35]:
X = df.drop("Purchased", axis=1)

y = df["Purchased"]

# ***Step 3: Train-Test Split***

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ***Step 4: Define Column Lists***

In [37]:
num_cols = ["Age", "Salary", "Experience"]

cat_cols = ["Gender", "City"]

### ***Why?***

- Because later ColumnTransformer will ask

> "Which columns should I send to the Numerical Pipeline?"

- and

>"Which columns should I send to the Categorical Pipeline?"

# ***Step 5: Create Numerical Pipeline***

In [38]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# ***Step 6: Create Categorical Pipeline***

In [39]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        drop="first",
        handle_unknown="ignore"
    ))
])

  
  
---

### Stop Here

Now we have created

```python
num_pipeline

cat_pipeline
```

Question:

Are these pipelines trained?

#### Answer

❌ No.

Exactly like

```python
model = LogisticRegression()
```

does not train the model,

```python
Pipeline(...)
```

also does not train anything.

It only creates the workflow.

---

### Current Architecture

```text
num_pipeline

Age
Salary
Experience

↓

Imputer

↓

Scaler
```

---

```text
cat_pipeline

Gender
City

↓

Imputer

↓

OneHotEncoder
```
---

# ***Step 7: Create ColumnTransformer***

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ]
)

# ***Step 12: Create the Main Pipeline***

In [41]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])

# ***Step 13: Train the Pipeline***

In [42]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Salary',
                                                   'Experience']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Gender', 'City'])])),
                ('model', LogisticRegression())])

> Scikit-learn automatically performs:

```
X_train
      │
      ▼
ColumnTransformer.fit()
      │
      ├───────────────┐
      │               │
      ▼               ▼
Numeric          Categorical
Pipeline          Pipeline
      │               │
      ▼               ▼
Imputer.fit()    Imputer.fit()
      │               │
      ▼               ▼
Scaler.fit()     Encoder.fit()
      └───────────────┘
              │
              ▼
Processed Data
              │
              ▼
LogisticRegression.fit()
```

# ***Step 14: Prediction***

In [44]:
predictions = pipeline.predict(X_test)

# ***Step 15: Check Accuracy***

In [45]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy : {accuracy * 100:.2f}%")

Accuracy : 48.70%


# ***Step 16: Save the Pipeline***

In [46]:
import pickle

with open("pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

# ***Step 17: Load the Pipeline***

In [47]:
with open("pipeline.pkl", "rb") as f:
    loaded_pipeline = pickle.load(f)

# ***Step 18: Create New Customer***

In [48]:
new_customer = pd.DataFrame({
    "Age": [30],
    "Salary": [50000],
    "Experience": [5],
    "Gender": ["Male"],
    "City": ["Delhi"]
})

# ***Step 19: Prediction***

In [49]:
prediction = loaded_pipeline.predict(new_customer)

print(prediction)

[1]


In [50]:
if prediction[0] == 1:
    print("Customer will Purchase")
else:
    print("Customer will Not Purchase")

Customer will Purchase


# ***Now See Real Difference***


---
##  ***Without Pipeline***

```text
New Customer
      │
      ▼
num_imputer.transform()
      │
      ▼
cat_imputer.transform()
      │
      ▼
encoder.transform()
      │
      ▼
Convert to DataFrame
      │
      ▼
Drop Columns
      │
      ▼
Concat
      │
      ▼
scaler.transform()
      │
      ▼
model.predict()
```

> Almost **20–30 lines**.

---

## ***With Pipeline***

```python
prediction = loaded_pipeline.predict(new_customer)
```

> **Only one line.** 

---

# ***Interview Question***

**Q. Why do we use Pipeline in Scikit-learn?**

**Answer:**

> **Pipeline combines preprocessing steps and the machine learning model into a single object. It automatically applies all preprocessing in the correct order during training and prediction, prevents data leakage, reduces manual code, and makes model deployment easier because only one pipeline object needs to be saved and loaded.**

---
